In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

target_model = AutoModelForCausalLM.from_pretrained("facebook/opt-350m", device_map="auto")
draft_model = AutoModelForCausalLM.from_pretrained("facebook/opt-125m", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m")
prompt = "Once upon a time there lived"
inputs = tokenizer(prompt, return_tensors="pt").to(target_model.device)
outputs = target_model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=1, assistant_model=draft_model)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/663M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/662M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'min_new_tokens', 'use_cache', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [2]:
result

'Once upon a time there lived somewhere a beautiful man named Charles who had lived in an estate, and he bought a garden, and to his delight he found it, that he should use every piece of the garden that he could get his hands on to develop it; as he went back to his apartment, and looked about the gardens and gardens, and the garden was being expanded. Now, Charles started working hard in those garden things, and soon, he found himself working in a garden garden.\n\nThis garden is known from'

In [ ]:
from transformers import pipeline

model_id = "google/bigbird-roberta-base"
pipeline = pipeline(task="fill-mask", model=model_id)
pipeline("She took some [MASK] medicine as she was feeling ill.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def angular_lsh(vectors, k):
    R = torch.randn(vectors.size(-1), k // 2, device=vectors.device)
    normalized_vectors = F.normalize(vectors, p=2.0, dim=1)
    V_proj = normalized_vectors @ R
    V_concat = torch.cat([V_proj, -V_proj], dim=1)
    return torch.argmax(V_concat, dim=1)

In [ ]:
torch.manual_seed(42)
vectors = torch.rand(16, 512)
angular_lsh(vectors, k=4)

In [ ]:
def phi(X, W):
    squared_norms = X.square().sum(dim=-1, keepdim=True)
    return torch.exp(X @ W - squared_norms / 2) / W.size(-1) ** 0.5

In [ ]:
def phi(X, W, dim_subtract_max=(-2, -1)):
    squared_norms = X.square().sum(dim=-1, keepdim=True)
    X_proj = X @ W
    max_vals = X_proj.amax(dim=dim_subtract_max, keepdim=True)
    return torch.exp(X_proj - max_vals - squared_norms / 2) / W.size(-1) ** 0.5

In [ ]:
class FavorAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_features):
        super().__init__()
        self.d_head = d_model // n_heads
        W = torch.randn(n_heads, self.d_head, n_features)  # h, d, m
        self.register_buffer("W", W)

    def forward(self, Q, K, V):
        scale = self.d_head ** -0.25
        Qp = phi(Q * scale, self.W, dim_subtract_max=-1)
        Kp = phi(K * scale, self.W)
        D = Qp @ Kp.sum(dim=-2).unsqueeze(-1)  # B, h, Lq, 1
        Kp_T_V = Kp.transpose(-2, -1) @ V      # B, h, m, d
        return (Qp @ Kp_T_V) / (D + 1e-6)

In [ ]:
def orthogonalize(W):
    d_head = W.size(-2)
    W_orth = torch.cat([torch.linalg.qr(W_chunk)[0]
                         for W_chunk in W.split(d_head, dim=-1)], dim=-1)
    return W_orth * d_head ** 0.5

In [ ]:
batch_size, Lq, Lk, d_head = 32, 100, 90, 64
n_heads = 8
n_groups = 2
query = torch.randn(batch_size, n_heads, Lq, d_head)
key = torch.randn(batch_size, n_groups, Lk, d_head)
value = torch.randn(batch_size, n_groups, Lk, d_head)
attn = F.scaled_dot_product_attention(query, key, value, enable_gqa=True)

In [ ]:
import peft
from transformers import AutoModelForCausalLM

model_id = "EleutherAI/gpt-neo-125M"  # a small model for this example
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto",
                                              dtype=torch.float16)
lora_config = peft.LoraConfig(r=8, target_modules=["q_proj", "v_proj"])
peft_model = peft.get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

In [ ]:
merged_model = peft_model.merge_and_unload()

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(output_dir="./my_great_model", num_train_epochs=3,
                          per_device_train_batch_size=8,
                          gradient_checkpointing=True, fp16=True)

In [ ]:
# [...] create the model, optimizer, criterion, and data_loader
accumulation_steps = 4
# optimizer.zero_grad()  # reset gradients before starting
# for batch_index, (X_batch, y_batch) in enumerate(data_loader):
#     X_batch, y_batch = X_batch.to(device), y_batch.to(device)
#     y_pred = model(X_batch)
#     loss = criterion(y_pred, y_batch)
#     loss = loss / accumulation_steps
#     loss.backward()
#     if (batch_index + 1) % accumulation_steps == 0:
#         optimizer.step()
#         optimizer.zero_grad()

In [ ]:
dp_model = torch.nn.DataParallel(model, device_ids=[0, 1, 2])
# [...] train dp_model normally